<a href="https://colab.research.google.com/github/marcos-mansur/load-forecast/blob/main/Data_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objective

Verify data quality
- identify missing days
- input with day before

# load libs

In [ ]:
!pip install pendulum

In [160]:
import tensorflow as tf
import numpy as np
import pandas as pd
import pendulum

#load data 

In [161]:
df_20XX = pd.read_csv(f'https://ons-dl-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2001.csv', 
                      sep=';', 
                      parse_dates=['din_instante'])

for x in range(2002,2022):
  df_20XX = pd.concat(objs = (df_20XX,pd.read_csv(f'https://ons-dl-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_{x}.csv', 
                      sep=';', 
                      parse_dates=['din_instante'])))

load_col = 'val_cargaenergiamwmed'
time_col = 'din_instante'

# minimum data treatment

In [164]:
def get_friday(date_time): 
  """ get next friday = start the operative week"""
  
  # today
  dt = pendulum.datetime(date_time.year,date_time.month, date_time.day)
  # return next friday
  return  dt.next(pendulum.FRIDAY).strftime('%Y-%m-%d')


def treat_data(df,regiao='SUDESTE',operative_week_start=2):
  
  df = df.copy()
  # round the values of load
  df['val_cargaenergiamwmed'] = np.round(df['val_cargaenergiamwmed'],2)
  # drop last 4 rows that doesn't have load values
  df.dropna(axis=0, how='any',inplace=True)
  # filter data by subsystem 
  try:
    df = df[df['nom_subsistema']==regiao].reset_index().drop('index',axis=1).copy()
  except:
    pass
  # dropa colunas sobre região
  df.drop(labels=['nom_subsistema','id_subsistema'], inplace=True, axis=1,errors='ignore')

  # create column with week number 
  df.reset_index(inplace=True,drop=True)
  df['semana'] = (df.index)//7 

  df['Mes'] = df['din_instante'].dt.month
  df['dia semana'] = df['din_instante'].dt.day_name()
  df['dia mes'] = df['din_instante'].dt.day
  df['ano'] = df['din_instante'].dt.year
  
  return df

df = treat_data(df_20XX, regiao='SUDESTE')
df.head(3)

,din_instante,val_cargaenergiamwmed,semana,Mes,dia semana,dia mes,ano
0,2001-01-01,19729.23,0,1,Monday,1,2001
1,2001-01-02,24596.20,0,1,Tuesday,2,2001
2,2001-01-03,26063.22,0,1,Wednesday,3,2001


# check missing data

In [169]:
print('Diff between numbers of days in the dataset for each year and 365: \n')
df.groupby(['ano'])[time_col].count() - [365 for x in range(2001,2022)]

Diff between numbers of days in the dataset for each year and 365: 



ano
2001    0
2002    0
2003    0
2004    1
2005    0
2006    0
2007    0
2008    1
2009    0
2010    0
2011    0
2012    1
2013    0
2014   -1
2015    0
2016   -8
2017    0
2018    0
2019    0
2020    1
2021    0
Name: din_instante, dtype: int64

There's one missing day in 2014 and 9 in 2016 (leap year).

In [171]:
# range of every day from 2001 to 2021
time_delta = pd.date_range(start = df.din_instante.iloc[0], end= df.din_instante.iloc[-1],freq='D')
# turn into df
df_time = pd.DataFrame(data={'data':time_delta})
# left join range of data with datas from dataset, missing days will become NaN
df_missing = df_time.join(df.set_index('din_instante'), on='data', how='left')
# missing days indexes
df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()]
missing_days = df_missing.loc[df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()].index].data
print('missing_days: \n',missing_days)

missing_days: 
 4779   2014-02-01
5573   2016-04-05
5574   2016-04-06
5575   2016-04-07
5576   2016-04-08
5577   2016-04-09
5578   2016-04-10
5579   2016-04-11
5580   2016-04-12
5581   2016-04-13
Name: data, dtype: datetime64[ns]


Let's impute the missing day in 2014 with the load from the day before and drop the incomplete weeks in 2016.

# fixed treat data

In [172]:
def get_missing_days(df):
  # range of every day from 2001 to 2021
  time_delta = pd.date_range(start = df.din_instante.iloc[0], end= df.din_instante.iloc[-1],freq='D')
  # turn into df
  df_time = pd.DataFrame(data={'data':time_delta})
  # left join range of data with datas from dataset, missing days will become NaN
  df_missing = df_time.join(df.set_index('din_instante'), on='data', how='left')
  # missing days indexes
  df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()]
  # series of missing days with indexes
  missing_days = df_missing.loc[df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()].index].data
  return missing_days

def id_and_impute(df):

  df = df.copy()
  # series of missing days with indexes
  missing_days = get_missing_days(df) 
  # drop days from incomplete week in 2016
  df = df.drop(axis=0, index = df.din_instante[(df[time_col]>='2016-04-01') & (df[time_col]<='2016-04-05')].index)
  df = df.drop(axis=0, index = missing_days.index[1]-1)
  # missing day to be inputed - feb 1st, 2014
  input_day = df['din_instante'].iloc[missing_days.index[0]]
  # line to be inserted in dataset with load value of day before
  df_day = pd.DataFrame({'din_instante': input_day-pd.Timedelta(1, unit='D'),	
                        'val_cargaenergiamwmed': df[load_col].iloc[missing_days.index[0] - 1]},
                        index=[missing_days.index[0] -1 ])
  # insert missing day
  df_total = pd.concat(objs= [df[:missing_days.index[0]], 
                              df_day, 
                              df[missing_days.index[0]:]])
  
  #reset index
  return  df_total, missing_days

def go_to_friday(df): 
  """ get next friday = start the operative week"""
  
  # first day in dataset
  date_time = df['din_instante'].iloc[0]
  # check if the dataset starts on a friday 
  if date_time.day_name() != 'Friday':
    # today
    dt = pendulum.datetime(date_time.year,date_time.month, date_time.day)
    # next friday - begins the operative week
    next_friday = dt.next(pendulum.FRIDAY).strftime('%Y-%m-%d')
    # df starts with the begin of operative week
    return df[df['din_instante'] >= next_friday].reset_index(drop=True).copy()


def treat_data(df,regiao='SUDESTE',operative_week_start=2):
  
  # round the values of load
  df['val_cargaenergiamwmed'] = np.round(df['val_cargaenergiamwmed'],2)
  # drop last 4 rows that doesn't have load values
  df.dropna(axis=0, how='any',inplace=True)
  # filter data by subsystem region
  try:
    df = df[df['nom_subsistema']==regiao].reset_index().drop('index',
                                                             axis=1).copy()
  except:
    pass
  # drops columns about region
  df.drop(labels=['nom_subsistema','id_subsistema'], 
          inplace=True, axis=1,errors='ignore')


  # check if the dataset starts on a friday and go to friday if it does not 
  df = go_to_friday(df)
  # insert missing data from 1st feb'2014
  df, missing_days = id_and_impute(df) 

  # create column with week number 
  df.reset_index(inplace=True,drop=True)
  df['semana'] = (df.index)//7 

  df['Mes'] = df['din_instante'].dt.month
  df['dia semana'] = df['din_instante'].dt.day_name()
  df['dia mes'] = df['din_instante'].dt.day
  df['ano'] = df['din_instante'].dt.year
  
  return df

df = treat_data(df_20XX, regiao='SUDESTE')
df.head(3)

,din_instante,val_cargaenergiamwmed,semana,Mes,dia semana,dia mes,ano
0,2001-01-05,26860.91,0,1,Friday,5,2001
1,2001-01-06,25047.63,0,1,Saturday,6,2001
2,2001-01-07,22943.49,0,1,Sunday,7,2001


In [173]:
print('Diff between numbers of days in the dataset for each year and 365: \n')
df.groupby(['ano'])[time_col].count() - [365 for x in range(2001,2022)]

Diff between numbers of days in the dataset for each year and 365: 



ano
2001    -4
2002     0
2003     0
2004     1
2005     0
2006     0
2007     0
2008     1
2009     0
2010     0
2011     0
2012     1
2013     0
2014     0
2015     0
2016   -13
2017     0
2018     0
2019     0
2020     1
2021     0
Name: din_instante, dtype: int64

-4 in 2001 was cut to begin the operative week on friday, nothing's wrong. 2016 is a leap year, so currently it has -14 days, two full operative weeks, that's fixed.
